# NLP Mini Project: Traditional Text Classification Pipeline

## Project Overview

In this project, students will build a **complete traditional Natural Language Processing (NLP) machine learning pipeline** for text classification. The focus is on **classical NLP techniques** rather than deep learning.

Students will learn how to:

- Perform text preprocessing
- Apply classical NLP techniques
- Convert text into numerical features
- Train machine learning models
- Evaluate model performance
- Save preprocessing objects and trained models
- Deploy the trained model using a simple FastAPI service

---

## Learning Objectives

By completing this project, students will:

1. Understand the steps of a traditional NLP pipeline
2. Implement text preprocessing techniques
3. Convert text into vector representations (BOW / TF-IDF)
4. Train classical ML models for text classification
5. Evaluate models using proper metrics
6. Save preprocessing pipelines and models
7. Deploy the model as an API using FastAPI

---

# Dataset


### Kaggle Dataset (Optional)

Students may alternatively use an easy Kaggle dataset such as:

Spam Classification

https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset

Task:

Classify SMS messages as:

- spam
- ham

---

# Project Requirements

Students must implement the following pipeline:

1. Data loading
2. Text preprocessing
3. Feature extraction
4. Model training
5. Model evaluation
6. Saving artifacts
7. API deployment

---

# Step 1 — Install Required Libraries

Students may need the following libraries:

- pandas
- numpy
- scikit-learn
- nltk
- spacy
- fastapi
- uvicorn
- joblib

Hint: Install missing libraries using pip.


In [2]:
!pip install pandas numpy scikit-learn nltk spacy fastapi uvicorn joblib

In [3]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 35.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# Step 2 — Load Dataset

### Option A: Load the Kaggle dataset

Hint:

Your task:

- Load the dataset
- Convert it into a pandas DataFrame
- Inspect sample rows

Questions:

- How many documents are there?
- How many classes exist?


In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'sms-spam-collection-dataset' dataset.
Path to dataset files: /kaggle/input/sms-spam-collection-dataset


In [5]:
import pandas as pd

url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"


df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

# Inspect
print(df.head())
print(f"\nTotal documents: {len(df)}")
print(f"\nNumber of classes: {df['label'].nunique()}")
print(f"Classes: {df['label'].unique()}")
print(f"\nClass distribution:\n{df['label'].value_counts()}")

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...

Total documents: 5572

Number of classes: 2
Classes: ['ham' 'spam']

Class distribution:
label
ham     4825
spam     747
Name: count, dtype: int64


# Step 3 — Text Preprocessing

Students must implement the following preprocessing steps:

### 1. Case Folding

Convert text to lowercase.

Example:

`"Hello World" → "hello world"`

---

### 2. Remove Punctuation and Numbers

Use **Regular Expressions (re)**.

Hint:

Use `re.sub()`.

---

### 3. Tokenization

Split text into tokens.

Example:

`"machine learning is fun" → ["machine","learning","is","fun"]`

---

### 4. Stopword Removal

Remove common words such as:

- the
- is
- and

Hint:

Use NLTK stopwords.

---

### 5. Stemming

Apply a stemming algorithm.

Example:

- running → run
- played → play

Hint:

Use `PorterStemmer`.

---

### 6. Lemmatization

Convert words to dictionary form.

Example:

- better → good
- running → run

Hint:

Use `WordNetLemmatizer`.

---

### Task

Create a **single preprocessing function** that performs all steps.


In [6]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize


nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

# tools
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    # Case Folding
    text = text.lower()

    # Remove Punctuation and Numbers
    text = re.sub(r'[^a-z\s]', '', text)

    # Tokenization
    tokens = word_tokenize(text)

    # Stopword Removal
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    # convert to string
    return ' '.join(tokens)

# dataset
df['cleaned'] = df['message'].apply(preprocess)

#  Result
print("Before:", df['message'][2])
print("After: ", df['cleaned'][2])

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Before: Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
After:  free entry wkly comp win fa cup final tkts st may text fa receive entry questionstd txt ratetcs apply over


# Step 4 — Feature Extraction

Convert text into numeric features.

Students must implement **two techniques**:

### 1. Bag of Words (BOW)

Hint:

`CountVectorizer`

---

### 2. TF-IDF

Hint:

`TfidfVectorizer`

---

Questions:

- What is the difference between BOW and TF-IDF?
- Which one performs better for your dataset?


In [7]:
from sklearn.feature_extraction.text import CountVectorizer

# BOW
bow_vectorizer = CountVectorizer()

X_bow = bow_vectorizer.fit_transform(df['cleaned'])

print("BOW shape:", X_bow.shape)


BOW shape: (5572, 7946)


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF
tfidf_vectorizer = TfidfVectorizer()

X_tfidf = tfidf_vectorizer.fit_transform(df['cleaned'])

print("TF-IDF shape:", X_tfidf.shape)


TF-IDF shape: (5572, 7946)


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# -------------------
# BOW Model
# -------------------
X_train_bow, X_test_bow, y_train, y_test = train_test_split(
    X_bow, df['label'], test_size=0.2, random_state=42
)

model_bow = MultinomialNB()
model_bow.fit(X_train_bow, y_train)

pred_bow = model_bow.predict(X_test_bow)

print("BOW Accuracy:", accuracy_score(y_test, pred_bow))


# -------------------
# TF-IDF Model
# -------------------
X_train_tfidf, X_test_tfidf, y_train, y_test = train_test_split(
    X_tfidf, df['label'], test_size=0.2, random_state=42
)

model_tfidf = MultinomialNB()
model_tfidf.fit(X_train_tfidf, y_train)

pred_tfidf = model_tfidf.predict(X_test_tfidf)

print("TF-IDF Accuracy:", accuracy_score(y_test, pred_tfidf))


BOW Accuracy: 0.968609865470852
TF-IDF Accuracy: 0.9713004484304932


# Step 5 — Train Machine Learning Models

Students must train **at least two models**.

Recommended models:

- Logistic Regression
- Naive Bayes
- Linear SVM

Hints:

Split the dataset using:

`train_test_split`

Train models on:

- BOW features
- TF-IDF features


In [10]:
from sklearn.model_selection import train_test_split

# BOW split
X_train_bow, X_test_bow, y_train, y_test = train_test_split(
    X_bow, df['label'], test_size=0.2, random_state=42
)

# TF-IDF split
X_train_tfidf, X_test_tfidf, y_train, y_test = train_test_split(
    X_tfidf, df['label'], test_size=0.2, random_state=42
)


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# BOW
lr_bow = LogisticRegression(max_iter=1000)
lr_bow.fit(X_train_bow, y_train)

pred_lr_bow = lr_bow.predict(X_test_bow)
print("Logistic Regression (BOW) Accuracy:", accuracy_score(y_test, pred_lr_bow))


# TF-IDF
lr_tfidf = LogisticRegression(max_iter=1000)
lr_tfidf.fit(X_train_tfidf, y_train)

pred_lr_tfidf = lr_tfidf.predict(X_test_tfidf)
print("Logistic Regression (TF-IDF) Accuracy:", accuracy_score(y_test, pred_lr_tfidf))


Logistic Regression (BOW) Accuracy: 0.9847533632286996
Logistic Regression (TF-IDF) Accuracy: 0.9596412556053812


In [12]:
from sklearn.naive_bayes import MultinomialNB

# BOW
nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train)

pred_nb_bow = nb_bow.predict(X_test_bow)
print("Naive Bayes (BOW) Accuracy:", accuracy_score(y_test, pred_nb_bow))


# TF-IDF
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)

pred_nb_tfidf = nb_tfidf.predict(X_test_tfidf)
print("Naive Bayes (TF-IDF) Accuracy:", accuracy_score(y_test, pred_nb_tfidf))


Naive Bayes (BOW) Accuracy: 0.968609865470852
Naive Bayes (TF-IDF) Accuracy: 0.9713004484304932


In [13]:
from sklearn.svm import LinearSVC

# BOW
svm_bow = LinearSVC()
svm_bow.fit(X_train_bow, y_train)

pred_svm_bow = svm_bow.predict(X_test_bow)
print("Linear SVM (BOW) Accuracy:", accuracy_score(y_test, pred_svm_bow))


# TF-IDF
svm_tfidf = LinearSVC()
svm_tfidf.fit(X_train_tfidf, y_train)

pred_svm_tfidf = svm_tfidf.predict(X_test_tfidf)
print("Linear SVM (TF-IDF) Accuracy:", accuracy_score(y_test, pred_svm_tfidf))


Linear SVM (BOW) Accuracy: 0.9847533632286996
Linear SVM (TF-IDF) Accuracy: 0.9820627802690582


# Step 6 — Model Evaluation

Students must evaluate models using:

- Accuracy
- Precision
- Recall
- F1 Score
- Confusion Matrix

Hint:

Use:

`classification_report`

and

`confusion_matrix`

---

### Analysis Questions

Students must answer:

- Which model performed best?
- Did TF-IDF outperform BOW?
- Why might that happen?


In [14]:
import pandas as pd
from sklearn.metrics import classification_report

# Dictionary of models and their predictions
models = {
    "Naive Bayes": pred_nb_tfidf,
    "Logistic Regression": pred_lr_tfidf,
    "Linear SVM": pred_svm_tfidf
}

# Empty DataFrame to store results
df_summary = pd.DataFrame()

for model_name, predictions in models.items():
    # Get classification report as a dictionary
    report_dict = classification_report(y_test, predictions, output_dict=True)

    # Take the weighted average metrics for the model
    avg_metrics = report_dict['weighted avg']

    # Add the model name
    avg_metrics['model'] = model_name

    # Append to the summary DataFrame
    df_summary = pd.concat([df_summary, pd.DataFrame([avg_metrics])])

# Arrange columns
cols = ['model', 'precision', 'recall', 'f1-score', 'support']
df_summary = df_summary[cols]

# Display the results
print(df_summary)



                 model  precision    recall  f1-score  support
0          Naive Bayes   0.972221  0.971300  0.969808   1115.0
0  Logistic Regression   0.960959  0.959641  0.956678   1115.0
0           Linear SVM   0.982110  0.982063  0.981628   1115.0


In [15]:
import pandas as pd
from sklearn.metrics import classification_report

# Dictionary of models and their predictions
models = {
    "Naive Bayes": pred_nb_bow,
    "Logistic Regression": pred_lr_bow,
    "Linear SVM": pred_svm_bow
}

# Empty DataFrame to store results
df_summary = pd.DataFrame()

for model_name, predictions in models.items():
    # Get classification report as a dictionary
    report_dict = classification_report(y_test, predictions, output_dict=True)

    # Take the weighted average metrics for the model
    avg_metrics = report_dict['weighted avg']

    # Add the model name
    avg_metrics['model'] = model_name

    # Append to the summary DataFrame
    df_summary = pd.concat([df_summary, pd.DataFrame([avg_metrics])])

# Arrange columns
cols = ['model', 'precision', 'recall', 'f1-score', 'support']
df_summary = df_summary[cols]

# Display the results
print(df_summary)


                 model  precision    recall  f1-score  support
0          Naive Bayes   0.971572  0.968610  0.969471   1115.0
0  Logistic Regression   0.985017  0.984753  0.984359   1115.0
0           Linear SVM   0.984872  0.984753  0.984408   1115.0


# Step 7 — Save Preprocessing and Models

Students must save the following objects:

- text preprocessing pipeline
- vectorizer
- trained model

Use either:

- pickle
- joblib

Example files:

```
preprocessor.pkl
vectorizer.pkl
model.pkl
```

Hints:

Saving models allows reuse without retraining.


In [17]:
import joblib

# ==============================
# 1️ Save Preprocessor
# ==============================
import pickle

with open('preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocess, f)

print(" Preprocessor saved!")

# ==============================
# 2️ Save Vectorizer
# ==============================
joblib.dump(tfidf_vectorizer, 'vectorizer.pkl')
print(" Vectorizer saved!")

# ==============================
# 3️ Save Model
# ==============================
joblib.dump(model, 'model.pkl')
print(" Model saved!")

 Preprocessor saved!
 Vectorizer saved!


NameError: name 'model' is not defined

# Step 8 — Build a FastAPI Model Service

Students will create a simple **prediction API**.

### Goal

Send text → receive predicted class.

---

## Project Structure

Suggested folder structure:

```
nlp_project
│
├── model.pkl
├── vectorizer.pkl
├── preprocess.pkl
│
├── app.py
└── notebook.ipynb
```

---

## FastAPI Implementation Hints

Steps:

1. Load saved objects
2. Create API endpoint
3. Accept input text
4. Apply preprocessing
5. Vectorize text
6. Predict label
7. Return prediction

---

### Example Endpoint

`POST /predict`

Input:

```
{
  "text": "Free money!!! Click this link"
}
```

Output:

```
{
  "prediction": "spam"
}
```

---

### Run the API

Students should run:

```
uvicorn app:app --reload
```

Then open:

http://127.0.0.1:8000/docs

to test the API using Swagger UI.


# Bonus Tasks (Optional)

Students can extend the project by:

- Creating a scikit-learn Pipeline
- Performing hyperparameter tuning
- Adding model comparison plots
- Deploying the API with Docker
- Building a simple Streamlit UI
